각 단어의 앞 두 글자(2-head word)의 빈도를 세세요. 그리고 빈도가 32 미만인 2-head word들의 빈도 합을 계산
하여 출력하세요


In [1]:
import os
import sys
import urllib.request
from pyspark.sql import SparkSession

# [핵심 해결책 1] 현재 실행 중인 파이썬 경로를 Spark 환경 변수에 강제 할당
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# [핵심 해결책 2] Spark 통신을 로컬호스트(127.0.0.1)로 명시하여 네트워크 타임아웃 방지
spark = SparkSession.builder \
    .appName("Q1_Sherlock") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

sc = spark.sparkContext

# 1. 파일 다운로드 및 RDD 로드
url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt"
urllib.request.urlretrieve(url, "sherlock.txt")

rdd = sc.textFile("sherlock.txt")

# 2. Spark 연산 파이프라인
under_32_rdd = rdd.flatMap(lambda line: line.split()) \
                  .filter(lambda w: len(w) >= 2) \
                  .map(lambda w: (w[:2].lower(), 1)) \
                  .reduceByKey(lambda a, b: a + b) \
                  .filter(lambda pair: pair[1] < 32) \
                  .map(lambda pair: pair[1])

# 3. 빈도 합계 계산
if not under_32_rdd.isEmpty():
    total_sum = under_32_rdd.reduce(lambda a, b: a + b)
    print("빈도가 32 미만인 2-head word들의 빈도 합:", total_sum)
else:
    print("조건을 만족하는 항목이 없습니다.")

빈도가 32 미만인 2-head word들의 빈도 합: 830
